# 02 Quality Review

Run this immediately after the quality audit. This notebook decides whether the dataset is clean enough for feature extraction and whether quality-filtered sensitivity analysis is needed.


In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = Path("/home/ubuntu/nishn_workspce/test_pdfs_generic/.covid_audio_btp_private/covid_audio_btp")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from covid_audio_btp.notebook_utils import (
    PROJECT_ROOT,
    artifact,
    check_artifacts,
    count_table,
    read_csv,
    read_optional_csv,
    require_artifacts,
    save_table,
    value_counts_frame,
    assert_no_participant_leakage,
    assert_binary_labels_present,
    stop_if_validation_errors,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
print(PROJECT_ROOT)


In [ ]:
require_artifacts(["data/processed/metadata_clean.csv", "data/processed/audio_quality.csv"])
metadata = read_csv("data/processed/metadata_clean.csv", n=5)
quality = read_csv("data/processed/audio_quality.csv", n=5)
merged = metadata.merge(quality, on=["recording_id", "audio_path"], how="left", suffixes=("_metadata", "_quality"))
if "quality_flag_quality" in merged.columns:
    merged["quality_flag"] = merged["quality_flag_quality"]
elif "quality_flag_metadata" in merged.columns:
    merged["quality_flag"] = merged["quality_flag_metadata"]
merged.head()


## Quality Gate


In [ ]:
summary = quality.groupby("quality_flag", dropna=False).agg(
    n=("recording_id", "count"),
    median_duration_sec=("duration_sec", "median"),
    median_silence_ratio=("silence_ratio", "median"),
    median_clipping_ratio=("clipping_ratio", "median"),
    median_event_duration_sec=("event_duration_sec", "median"),
).reset_index().sort_values("n", ascending=False)
summary["percent"] = summary["n"] / max(len(quality), 1)
save_table(summary, "reports/tables/nb02_quality_summary.csv")
display(summary)

ok_rate = float((quality["quality_flag"] == "ok").mean()) if len(quality) else 0.0
corrupt_rate = float((quality["quality_flag"] == "corrupt").mean()) if len(quality) else 1.0
print(f"ok_rate={ok_rate:.3f}, corrupt_rate={corrupt_rate:.3f}")
if corrupt_rate > 0.05:
    raise AssertionError("Corrupt audio rate is above 5%. Inspect files before feature extraction.")
if ok_rate < 0.50:
    raise AssertionError("Less than half the recordings are quality=ok. Do not train until reviewed.")


## Quality By Modality And Label


In [ ]:
quality_by_group = count_table(merged, ["modality", "label_binary", "quality_flag"])
save_table(quality_by_group, "reports/tables/nb02_quality_by_modality_label.csv")
display(quality_by_group)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.countplot(data=merged, x="modality", hue="quality_flag", ax=axes[0])
axes[0].set_title("Quality flags by modality")
sns.boxplot(data=merged, x="modality", y="event_duration_sec", hue="label_binary", ax=axes[1])
axes[1].set_title("Active event duration by modality")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "reports/figures/nb02_quality_by_modality.png", dpi=160)
plt.show()


## Bad Sample Tables


In [ ]:
bad = merged[merged["quality_flag"] != "ok"].copy()
bad_cols = [
    "recording_id", "participant_id", "modality", "submodality", "label_binary",
    "quality_flag", "quality_reasons", "duration_sec_quality", "duration_sec",
    "silence_ratio", "clipping_ratio", "audio_path",
]
bad_cols = [c for c in bad_cols if c in bad.columns]
save_table(bad[bad_cols].head(200), "reports/tables/nb02_bad_samples_top200.csv")
bad[bad_cols].head(30)


## Numeric Quality Distributions


In [ ]:
numeric_cols = ["duration_sec", "silence_ratio", "clipping_ratio", "snr_proxy", "active_audio_ratio", "event_duration_sec"]
available_numeric = [c for c in numeric_cols if c in quality.columns]
display(quality[available_numeric].describe().T)

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
axes = axes.ravel()
for ax, col in zip(axes, available_numeric):
    sns.histplot(data=quality, x=col, hue="quality_flag", bins=40, ax=ax, element="step")
    ax.set_title(col)
for ax in axes[len(available_numeric):]:
    ax.axis("off")
fig.tight_layout()
fig.savefig(PROJECT_ROOT / "reports/figures/nb02_quality_numeric_distributions.png", dpi=160)
plt.show()
